# Démonstration : BERT

Ce notebook permet de :

- charger un modèle de type BERT ;
- tokeniser une ou plusieurs phrases ;
- observer les tokens produits par le tokenizer ;
- obtenir des embeddings contextualisés ;
- comparer rapidement le même mot dans deux contextes différents.

## 1. Préparer l'environnement

Nous allons utiliser deux bibliothèques principales :

- `transformers`, pour charger le modèle et le tokenizer ;
- `torch`, pour manipuler les tenseurs produits par le modèle.

Le modèle sera téléchargé depuis Hugging Face lors de la première exécution.  

In [1]:
# Si nécessaire, installer les bibliothèques.
# !pip install transformers torch pandas numpy

try:
    import torch
    import numpy as np
    import pandas as pd
    from transformers import AutoTokenizer, AutoModel
except ImportError as e:
    raise ImportError(
        "Il manque une bibliothèque. Installez-la avec : "
        "pip install transformers torch pandas numpy"
    ) from e

torch.set_grad_enabled(False)

## 2. Choisir et charger un modèle

Pour la démo, on utilise un modèle multilingue de type BERT.

Il est possible de remplacer le nom du modèle par un autre modèle compatible avec Hugging Face, par exemple :

- `bert-base-multilingual-cased`
- `distilbert-base-multilingual-cased`
- un modèle local déjà téléchargé

In [2]:
MODEL_NAME = "distilbert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME, output_attentions=True)
model.eval()

print(f"Modèle chargé : {MODEL_NAME}")
print(f"Taille du vocabulaire du tokenizer : {tokenizer.vocab_size}")

Modèle chargé : distilbert-base-multilingual-cased
Taille du vocabulaire du tokenizer : 119547


## 3. Une première phrase

Commençons avec une phrase simple.

Le tokenizer va transformer cette phrase en identifiants numériques.  
Le modèle ne reçoit pas directement des mots : il reçoit des nombres.

In [3]:
phrase = "Le roi donne son épée à son fils parce qu'il part à la guerre."

encoded = tokenizer(phrase, return_tensors="pt")

encoded

{'input_ids': tensor([[  101, 10281, 15681, 17717, 10312,   263, 45249,   254, 10312, 15017,
         46718, 10608,   112, 10154, 10668,   254, 10109, 14158,   119,   102]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

## 4. Observer les tokens

Les identifiants numériques sont peu lisibles pour nous.

On peut donc demander au tokenizer de reconvertir ces identifiants en tokens.

In [4]:
def afficher_tokens(text):
    encoded = tokenizer(text, return_tensors="pt")
    tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])
    return encoded, pd.DataFrame({
        "position": range(len(tokens)),
        "token": tokens,
        "id": encoded["input_ids"][0].tolist(),
        "attention_mask": encoded["attention_mask"][0].tolist()
        })

encoded, df_tokens = afficher_tokens(phrase)
df_tokens

,position,token,id,attention_mask
0,0,[CLS],101,1
1,1,Le,10281,1
2,2,roi,15681,1
3,3,donne,17717,1
4,4,son,10312,1
5,5,é,263,1
6,6,##pée,45249,1
7,7,à,254,1
8,8,son,10312,1
9,9,fils,15017,1


In [5]:
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])

tokens

['[CLS]',
 'Le',
 'roi',
 'donne',
 'son',
 'é',
 '##pée',
 'à',
 'son',
 'fils',
 'parce',
 'qu',
 "'",
 'il',
 'part',
 'à',
 'la',
 'guerre',
 '.',
 '[SEP]']

## 5. Que signifient les tokens spéciaux ?

In [6]:
print("Token de début :", tokenizer.cls_token)
print("Token de séparation :", tokenizer.sep_token)
print("Token de masque :", tokenizer.mask_token)

Token de début : [CLS]
Token de séparation : [SEP]
Token de masque : [MASK]


## 6. Passer la phrase dans le modèle

Nous pouvons maintenant envoyer les tokens au modèle.

Le modèle produit un tenseur appelé `last_hidden_state`.

Ce tenseur contient une représentation vectorielle pour chaque token de la phrase.

In [7]:
with torch.no_grad():
    outputs = model(**encoded)

last_hidden_state = outputs.last_hidden_state

print("Forme du tenseur :", last_hidden_state.shape)

Forme du tenseur : torch.Size([1, 20, 768])


## 7. Lire la forme du tenseur

La forme du tenseur suit généralement ce format :

```text
(nombre de phrases, nombre de tokens, taille des embeddings)
```

Par exemple :

```text
(1, 22, 768)
```

signifie :

- 1 phrase ;
- 22 tokens ;
- un vecteur de 768 dimensions pour chaque token.

Le modèle ne produit donc pas un seul nombre.  
Il produit une représentation dense pour chaque token.

In [8]:
nb_phrases, nb_tokens, taille_embedding = last_hidden_state.shape

print(f"Nombre de phrases : {nb_phrases}")
print(f"Nombre de tokens : {nb_tokens}")
print(f"Taille de chaque embedding : {taille_embedding}")

Nombre de phrases : 1
Nombre de tokens : 20
Taille de chaque embedding : 768


## 8. Voir l'embedding d'un token

Prenons l'embedding du premier vrai token après `[CLS]`.

C'est un vecteur numérique.  
Il peut être utilisé pour des comparaisons, des classifications ou d'autres traitements.

In [9]:
position = 1
token = tokens[position]
embedding = last_hidden_state[0, position]

print("Token :", token)
print("Taille de l'embedding :", embedding.shape)
print("Premières valeurs :")
embedding[:10]

Token : Le
Taille de l'embedding : torch.Size([768])
Premières valeurs :


tensor([ 0.1787, -0.4643, -0.1370,  0.0980,  0.4552,  0.4006, -0.1277,  0.6447,
        -0.1096,  1.1765])

## 9. Même mot, deux contextes

BERT produit des embeddings contextualisés.

Cela signifie que le même mot peut recevoir une représentation différente selon la phrase.

In [10]:
phrases_tombe = {
    "verbe": "Le roi tombe au début du récit.",
    "nom": "La tombe du roi se trouve près du temple.",
    "nom_2": "Une tombe ancienne se trouve près du temple."
}

for etiquette, texte in phrases_tombe.items():
    _, tableau = afficher_tokens(texte)
    print("" + etiquette.upper())
    display(tableau)

VERBE


,position,token,id,attention_mask
0,0,[CLS],101,1
1,1,Le,10281,1
2,2,roi,15681,1
3,3,tombe,40921,1
4,4,au,10257,1
5,5,début,15214,1
6,6,du,10168,1
7,7,récit,66930,1
8,8,.,119,1
9,9,[SEP],102,1


NOM


,position,token,id,attention_mask
0,0,[CLS],101,1
1,1,La,10159,1
2,2,tombe,40921,1
3,3,du,10168,1
4,4,roi,15681,1
5,5,se,10126,1
6,6,trouve,15969,1
7,7,près,16092,1
8,8,du,10168,1
9,9,temple,19603,1


NOM_2


,position,token,id,attention_mask
0,0,[CLS],101,1
1,1,Une,13509,1
2,2,tombe,40921,1
3,3,ancienne,20201,1
4,4,se,10126,1
5,5,trouve,15969,1
6,6,près,16092,1
7,7,du,10168,1
8,8,temple,19603,1
9,9,.,119,1


## 10. Extraire l'embedding d'un mot

Un mot peut parfois être découpé en plusieurs tokens.

La fonction suivante cherche les tokens correspondant à un mot cible, puis calcule la moyenne de leurs embeddings.

In [11]:
def trouver_sous_sequence(sequence, sous_sequence):
    positions = []
    n = len(sous_sequence)
    for i in range(len(sequence) - n + 1):
        if sequence[i:i+n] == sous_sequence:
            positions.append(i)
    return positions


def embedding_du_mot(text, word):
    encoded = tokenizer(text, return_tensors="pt")
    tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])
    word_tokens = tokenizer.tokenize(word)
    
    positions = trouver_sous_sequence(tokens, word_tokens)
    
    if not positions:
        raise ValueError(
            f"Impossible de trouver {word!r} dans les tokens. "
            f"Tokens du mot : {word_tokens}. Tokens de la phrase : {tokens}"
        )
    
    start = positions[0]
    end = start + len(word_tokens)
    
    with torch.no_grad():
        outputs = model(**encoded)
    
    # Si le mot est découpé en plusieurs sous-mots, on fait la moyenne.
    embedding = outputs.last_hidden_state[0, start:end].mean(dim=0)
    
    return {
        "texte": text,
        "mot": word,
        "tokens_phrase": tokens,
        "tokens_mot": word_tokens,
        "position_debut": start,
        "position_fin": end - 1,
        "embedding": embedding
    }


resultats_tombe = {
    etiquette: embedding_du_mot(texte, "tombe")
    for etiquette, texte in phrases_tombe.items()
}

for etiquette, resultat in resultats_tombe.items():
    print(etiquette, "→", resultat["tokens_mot"], "position :", resultat["position_debut"])

verbe → ['tombe'] position : 3
nom → ['tombe'] position : 2
nom_2 → ['tombe'] position : 2


## 11. Comparer les embeddings

Pour comparer deux embeddings, on peut utiliser la similarité cosinus.

- Une valeur proche de 1 indique des vecteurs très proches.
- Une valeur proche de 0 indique des vecteurs plus indépendants.

In [12]:
from torch.nn.functional import cosine_similarity

def sim_cos(a, b):
    return cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()

comparaisons = [
    ("tombe verbe", "tombe nom", resultats_tombe["verbe"]["embedding"], resultats_tombe["nom"]["embedding"]),
    ("tombe nom", "tombe nom_2", resultats_tombe["nom"]["embedding"], resultats_tombe["nom_2"]["embedding"]),
    ("tombe verbe", "tombe nom_2", resultats_tombe["verbe"]["embedding"], resultats_tombe["nom_2"]["embedding"]),
]

pd.DataFrame([
    {
        "comparaison": f"{a} / {b}",
        "similarité_cosinus": sim_cos(emb_a, emb_b)
    }
    for a, b, emb_a, emb_b in comparaisons
])

,comparaison,similarité_cosinus
0,tombe verbe / tombe nom,0.700430
1,tombe nom / tombe nom_2,0.893020
2,tombe verbe / tombe nom_2,0.644084


## 12. Une représentation de phrase

On peut aussi produire une représentation grossière d'une phrase en faisant la moyenne des embeddings des tokens.

In [13]:
def embedding_de_phrase(text):
    encoded = tokenizer(text, return_tensors="pt")
    
    with torch.no_grad():
        outputs = model(**encoded)
    
    token_embeddings = outputs.last_hidden_state
    attention_mask = encoded["attention_mask"].unsqueeze(-1)
    
    # On ignore les positions masquées avec attention_mask.
    phrase_embedding = (token_embeddings * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
    
    return phrase_embedding[0]


phrases = [
    "Ce texte parle de deuil.",
    "Ce texte parle de mort.",
    "Ce texte parle d'agriculture."
]

embeddings_phrases = [embedding_de_phrase(p) for p in phrases]

matrice = np.zeros((len(phrases), len(phrases)))

for i in range(len(phrases)):
    for j in range(len(phrases)):
        matrice[i, j] = sim_cos(embeddings_phrases[i], embeddings_phrases[j])

pd.DataFrame(matrice, index=phrases, columns=phrases)

,Ce texte parle de deuil.,Ce texte parle de mort.,Ce texte parle d'agriculture.
Ce texte parle de deuil.,1.000000,0.927246,0.853848
Ce texte parle de mort.,0.927246,1.000000,0.846684
Ce texte parle d'agriculture.,0.853848,0.846684,1.000000
